Лабораторная раота №1

Kaggle: https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates

В этом блокноте будем проводить очистку датасета и последубщее сохранение в айсберг

1) Инициализация спарк

In [3]:
from src.spark_session import create_spark

spark = create_spark("Lab_1_Data_Cleaning")
spark

Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true
Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/ivy/cache
The jars for the packages stored in: /tmp/ivy/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1cad4da8-0b8e-4074-8f73-9e46484ccac4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
:: resolution report :: resolve 251ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-p

2) Пути и имя таблицы

In [4]:
INPUT_PATH = "/app/data/raw/steam_games.csv"
TABLE_NAME = "local.lab1.steam_games"

ROW_LIMIT = None

print(INPUT_PATH)
print(TABLE_NAME)
print(ROW_LIMIT)

/app/data/raw/steam_games.csv
local.lab1.steam_games
None


3) Чтение датасета

In [5]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(INPUT_PATH)
)

if ROW_LIMIT is not None:
    df_raw = df_raw.limit(ROW_LIMIT)

print("Названия столбцов:", df_raw.columns)
print("Количество столбцов:", len(df_raw.columns))
print("Количество партиций:", df_raw.rdd.getNumPartitions())

df_raw.printSchema()

Названия столбцов: ['app_id', 'name', 'release_date', 'price', 'price_status', 'estimated_owners', 'developers', 'publishers', 'genres', 'categories', 'tags', 'positive', 'negative', 'recommendations', 'peak_ccu', 'metacritic_score', 'user_score', 'average_playtime_forever', 'median_playtime_forever', 'average_playtime_2weeks', 'median_playtime_2weeks', 'short_description', 'about_the_game', 'detailed_description', 'notes', 'achievements', 'dlc_count', 'packages', 'supported_languages', 'full_audio_languages', 'windows', 'mac', 'linux', 'header_image', 'screenshots', 'movies', 'website', 'support_url', 'support_email', 'metacritic_url', 'steam_store_available', 'steam_spy_available']
Количество столбцов: 42
Количество партиций: 8
root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_status: string (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- dev

4) Глянем первые 20 строк в датасете

In [6]:
df_raw.show(10, truncate=False)

26/09/12 09:30:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+--------------------+------------+-----+------------+----------------+-----------------------------+-----------------------------+---------------+--------------+-------------------+---------------------+-------------------------------+-----------------------+-----------------------+----------------------------+-------------------------------+-------------------------------+-----------------------+-----------------------+----------------------+------------------+--------------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

5) Выберем ниболее интересующие колонки

In [7]:
SELECTED_COLUMNS = [
    "app_id",
    "name",
    "release_date",
    "price",
    "price_status",
    "estimated_owners",
    "developers",
    "publishers",
    "genres",
    "categories",
    "tags",
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    "user_score",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks",
    "achievements",
    "dlc_count",
    "supported_languages",
    "full_audio_languages",
    "windows",
    "mac",
    "linux",
    "steam_store_available",
    "steam_spy_available"
]

df = df_raw.select(*SELECTED_COLUMNS)

df.show(10, truncate=False)

+------+--------------------+------------+-----+------------+----------------+-----------------------------+-----------------------------+---------------+--------------+-------------------+---------------------+-------------------------------+-----------------------+-----------------------+----------------------------+-------------------------------+-------------------------------+-----------------------+-----------------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------

6) Сохраним отдельный csv и глянем данные внутри

In [8]:
from pathlib import Path

INPUT_PATH = "/app/data/raw/steam_games.csv"
OUTPUT_PATH = "/app/output/results/steam_sample_1000.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(INPUT_PATH)
)

# В исходном CSV первый столбец с App ID имеет пустой заголовок.
# Spark может назвать его, например, _c0.
first_column = df_raw.columns[0]

if first_column != "app_id":
    df_raw = df_raw.withColumnRenamed(first_column, "app_id")

columns = [
    "app_id",
    "name",
    "release_date",
    "price",
    "price_status",
    "estimated_owners",
    "developers",
    "publishers",
    "genres",
    "categories",
    "tags",
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    "user_score",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks",
    "achievements",
    "dlc_count",
    "supported_languages",
    "full_audio_languages",
    "windows",
    "mac",
    "linux",
    "steam_store_available",
    "steam_spy_available",
]

df_sample = (
    df_raw
    .select(*columns)
    .limit(1000)
)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

df_sample.toPandas().to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(f"CSV сохранён: {OUTPUT_PATH}")
print(f"Строк: {df_sample.count()}")
print(f"Столбцов: {len(df_sample.columns)}")

CSV сохранён: /app/output/results/steam_sample_1000.csv


Строк: 1000
Столбцов: 30
